In [1]:
import os
from os.path import expanduser
home = expanduser("~/")

import sys
# sys.path.insert(0, '/global/u2/x/xshuang/gigalens-xh-dev/src')

# import sys
conda_env = sys.path[1]
del sys.path[1]

import os
# sys.path.append(f'{os.environ['HOME']}/gigalens_personal/gigalens/src')
sys.path.append(home+'/gigalens'+'/src')
sys.path.append(conda_env)
print(sys.path)



srcdir = os.path.join(home, "gigalens/src/")


['', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '/usr/local/lib/python3.12/dist-packages', '__editable__.flax-0.10.6.finder.__path_hook__', '__editable__.jax_cuda12_pjrt-0.6.2.dev20250607.finder.__path_hook__', '__editable__.jax_cuda12_plugin-0.6.2.dev20250607.finder.__path_hook__', '__editable__.nsys_jax-0.1.dev1163+g4a8f06a.finder.__path_hook__', '/opt/pip/src', '/usr/lib/python3/dist-packages', '/global/homes/l/linusu//gigalens/src', '/global/homes/l/linusu/.conda/envs/gigalens_multinode_env/lib/python3.12/site-packages']


In [2]:
import tensorflow_probability.substrates.jax as tfp

from gigalens.jax.inference import ModellingSequence
from gigalens.jax.model import ForwardProbModel, BackwardProbModel
from gigalens.model import PhysicalModel
from gigalens.jax.simulator import LensSimulator
from gigalens.simulator import SimulatorConfig
from gigalens.jax.profiles.light import sersic
from gigalens.jax.profiles.mass import epl, shear

import jax
from jax import random
import numpy as np
import optax
from jax import numpy as jnp
from matplotlib import pyplot as plt
import optax
import corner
import yaml
import pickle
from helpers import *
import blackjax
tfd = tfp.distributions

2025-12-16 19:47:23.600912: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765943243.611517 1403742 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765943243.615989 1403742 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1765943243.627926 1403742 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765943243.627940 1403742 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765943243.627941 1403742 computation_placer.cc:177] computation placer alr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

/global/homes/l/linusu/.conda/envs/gigalens_multinode_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/global/homes/l/linusu//gigalens/src/gigalens/jax/inference.py:28: UserWarning: jax.distributed.initialize() was not called. For multinode, please call it before running any JAX functions.
  warnings.warn('jax.distributed.initialize() was not called. For multinode, please call it before running any JAX functions.')


In [3]:
prior = make_default_prior()
gigal_dir = os.path.join(home,'gigalens/src/gigalens')
kernel = np.load(gigal_dir + '/assets/psf.npy').astype(np.float32)
sim_config = SimulatorConfig(delta_pix=0.065, num_pix=60, supersample=2, kernel=kernel)
phys_model = PhysicalModel([epl.EPL(50), shear.Shear()], [sersic.SersicEllipse(use_lstsq=False)], [sersic.SersicEllipse(use_lstsq=False)])
lens_sim = LensSimulator(phys_model, sim_config, bs=1)
observed_img = np.load(gigal_dir + '/assets/demo.npy')
prob_model = ForwardProbModel(prior, observed_img, background_rms=0.2, exp_time=100)
model_seq = ModellingSequence(phys_model, prob_model, sim_config)

In [4]:
cfg = PipelineConfig(steps=["MAP", "SVI", "HMC"], map_kwargs=dict(num_steps=350, n_samples=500),
    svi_kwargs=dict(num_steps=1500, n_vi=500))

results = run_pipeline(model_seq, cfg)

Starting MAP
Starting SVI


2025-12-16 19:48:28.142848: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.


Starting HMC


In [5]:
def run_mclmc(logdensity_fn, num_steps, initial_position, precond_inv_mass_matrix, key, transform, desired_energy_variance= 5e-4):
    init_key, tune_key, run_key = jax.random.split(key, 3)

    # create an initial state for the sampler
    initial_state = blackjax.mcmc.mclmc.init(
        position=initial_position, logdensity_fn=logdensity_fn, rng_key=init_key
    )

    # build the kernel
    kernel = lambda inverse_mass_matrix : blackjax.mcmc.mclmc.build_kernel(
        logdensity_fn=logdensity_fn,
        integrator=blackjax.mcmc.integrators.isokinetic_mclachlan,
        inverse_mass_matrix=inverse_mass_matrix,
    )
    dim =22
    starting_adapt_state = blackjax.adaptation.mclmc_adaptation.MCLMCAdaptationState(
            jnp.sqrt(dim), jnp.sqrt(dim) * 0.25, inverse_mass_matrix=precond_inv_mass_matrix
    )
    # find values for L and step_size
    (
        blackjax_state_after_tuning,
        blackjax_mclmc_sampler_params,
        _
    ) = blackjax.mclmc_find_L_and_step_size(
        mclmc_kernel=kernel,
        num_steps=num_steps,
        state=initial_state,
        rng_key=tune_key,
        diagonal_preconditioning=False,
        params=starting_adapt_state,
        desired_energy_var=desired_energy_variance
    )

    # use the quick wrapper to build a new kernel with the tuned parameters
    sampling_alg = blackjax.mclmc(
        logdensity_fn,
        L=blackjax_mclmc_sampler_params.L,
        step_size=blackjax_mclmc_sampler_params.step_size,
    )

    # run the sampler
    _, samples = blackjax.util.run_inference_algorithm(
        rng_key=run_key,
        initial_state=blackjax_state_after_tuning,
        inference_algorithm=sampling_alg,
        num_steps=num_steps,
        transform=transform,
        progress_bar=True,
    )

    return samples, blackjax_state_after_tuning, blackjax_mclmc_sampler_params, run_key


In [6]:

rng_key = jax.random.key(0)

logdensity_fn = lambda x: -0.5 * jnp.sum(jnp.square(x))
num_steps = 10000
transform = lambda state, info: state.position

sample_key, rng_key = jax.random.split(rng_key)
samples, initial_state, params, chain_key = run_mclmc(
    logdensity_fn=logdensity_fn,
    num_steps=num_steps,
    initial_position=jnp.ones((1000,)),
    precond_inv_mass_matrix=1,
    key=sample_key,
    transform=transform,
)
samples.mean()

Array(0.00013477, dtype=float32)

In [7]:


num_steps = 1000
transform = lambda state, info: state.position
sample_key, rng_key = jax.random.split(rng_key)
samples, initial_state, params, chain_key = run_mclmc(
    logdensity_fn=log_prob,
    num_steps=num_steps,
    initial_position=start,
    precond_inv_mass_matrix=jnp.diag(inv_mass_mat),
    key=sample_key,
    transform=transform,
)

NameError: name 'log_prob' is not defined

In [ ]:
# init_key, tune_key, run_key = jax.random.split(rng_key, 3)

# sampling_alg = blackjax.mclmc(
#     logdensity_fn,
#     L=0.01,
#     step_size=0.1,
#     inverse_mass_matrix=jnp.diag(inv_mass_mat)
# )

# initial_state = blackjax.mcmc.mclmc.init(
#     position=start, logdensity_fn=logdensity_fn, rng_key=init_key
# )

# _, samples = blackjax.util.run_inference_algorithm(
#     rng_key=run_key,
#     initial_state=initial_state,
#     inference_algorithm=sampling_alg,   
#     num_steps=300000,
#     transform=transform,
#     progress_bar=True,
# )

In [9]:
def log_prob(z):
    return prob_model.log_prob(lens_sim, z)[0]

start = jnp.squeeze(results['SVI'].qz.mean())
inv_mass_mat = results['SVI'].qz.covariance()
transform = lambda state, info: state.position

In [15]:
from blackjax.mcmc.adjusted_mclmc_dynamic import rescale
rng_key = jax.random.key(0)

init_key, tune_key, run_key = jax.random.split(rng_key, 3)

integration_steps_fn = lambda avg_num_integration_steps: lambda _: jnp.ceil(avg_num_integration_steps)

kernel = lambda rng_key, state, avg_num_integration_steps, step_size, inverse_mass_matrix: blackjax.mcmc.adjusted_mclmc_dynamic.build_kernel(
    integration_steps_fn=integration_steps_fn(avg_num_integration_steps),
    inverse_mass_matrix=inverse_mass_matrix,
)(
    rng_key=rng_key,
    state=state,
    step_size=step_size,
    logdensity_fn=log_prob,
    L_proposal_factor=jnp.inf,
)
initial_state = blackjax.mcmc.adjusted_mclmc_dynamic.init(
    position=start, logdensity_fn=log_prob, random_generator_arg=init_key
)
dim=22
starting_adapt_state = blackjax.adaptation.adjusted_mclmc_adaptation.MCLMCAdaptationState(
            jnp.sqrt(dim), jnp.sqrt(dim) * 0.25, inverse_mass_matrix=jnp.diag(inv_mass_mat),
    )

# find values for L and step_size
(
    blackjax_state_after_tuning,
    blackjax_mclmc_sampler_params,
    _
) = blackjax.adjusted_mclmc_find_L_and_step_size(
    mclmc_kernel=kernel,
    num_steps=1000,
    state=initial_state,
    rng_key=tune_key,
    target=0.9,
    diagonal_preconditioning=False,
    params=starting_adapt_state,
    # desired_energy_var=desired_energy_variance
)

step_size = blackjax_mclmc_sampler_params.step_size
L = blackjax_mclmc_sampler_params.L



# _, samples = blackjax.util.run_inference_algorithm(
#     rng_key=run_key,
#     initial_state=initial_state,
#     inference_algorithm=sampling_alg,   
#     num_steps=1000,
#     transform=transform,
#     progress_bar=True,
# )

In [16]:
sampling_alg = blackjax.adjusted_mclmc_dynamic(
    logdensity_fn=logdensity_fn,
    step_size=step_size,
    integration_steps_fn=lambda key: jnp.ceil(
        jax.random.uniform(key) * rescale(L / step_size)
    ),
    inverse_mass_matrix=blackjax_mclmc_sampler_params.inverse_mass_matrix,
    L_proposal_factor=jnp.inf,
)

_, out = blackjax.util.run_inference_algorithm(
    rng_key=run_key,
    initial_state=blackjax_state_after_tuning,
    inference_algorithm=sampling_alg,
    num_steps=1000,
    transform=transform,
    progress_bar=True,
)

In [17]:
blackjax_mclmc_sampler_params

MCLMCAdaptationState(L=Array(0.93699807, dtype=float32), step_size=Array(0.7137708, dtype=float32), inverse_mass_matrix=Array([3.4088735e-06, 3.0725305e-06, 2.0100229e-05, 1.5847500e-05,
       2.0944790e-03, 1.0228040e-06, 4.7403364e-06, 3.4079678e-06,
       5.9075115e-05, 3.2620872e-05, 6.0515697e-07, 4.5944017e-07,
       3.3063851e-03, 1.9134758e-01, 7.5747004e-05, 1.7941331e-03,
       9.2929619e-04, 1.8520894e-06, 2.7965064e-06, 1.6776933e-03,
       1.5965252e-03, 1.2936345e-03], dtype=float32))

In [24]:
blackjax_mclmc_sampler_params.inverse_mass_matrix

Array([3.4088735e-06, 3.0725305e-06, 2.0100229e-05, 1.5847500e-05,
       2.0944790e-03, 1.0228040e-06, 4.7403364e-06, 3.4079678e-06,
       5.9075115e-05, 3.2620872e-05, 6.0515697e-07, 4.5944017e-07,
       3.3063851e-03, 1.9134758e-01, 7.5747004e-05, 1.7941331e-03,
       9.2929619e-04, 1.8520894e-06, 2.7965064e-06, 1.6776933e-03,
       1.5965252e-03, 1.2936345e-03], dtype=float32)

In [23]:
# print("L:", params.L, "Step Size:", params.step_size)
print(np.abs(blackjax_mclmc_sampler_params.inverse_mass_matrix - jnp.diag(inv_mass_mat))/jnp.diag(inv_mass_mat))

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [20]:
MCMC_x = prob_model.bij.forward(list(samples.T))
fig = cornerplot_posterior(MCMC_x, color='red')
cornerplot_posterior(results['HMC'].HMC_samples, fig=fig, color='black')
cornerplot_posterior(results['SVI'].SVI_samples, fig=fig, color='blue')
plt.show()

ValueError: It looks like the parameter(s) in column(s) 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21 have no dynamic range. Please provide a `range` argument.

Error in callback <function flush_figures at 0x7effac3f47c0> (for post_execute), with arguments args (),kwargs {}:


KeyboardInterrupt: 

In [ ]:
tfp.mcmc.effective_sample_size(samples)/10000

In [ ]:
tfp.mcmc.effective_sample_size(results['HMC'].HMC_samples_z[0, 0, 0])/750

In [ ]:
results['HMC'].HMC_samples_z.shape

In [ ]:
4 * 12 * 750